# RASE (v2, revised): Regime-Aware Stacked Ensemble for AQI Forecasting in 63 Indian Cities
**Adbhut Prakash Dwivedi, Raghuraj Singh** - Harcourt Butler Technical University, Kanpur

This notebook reproduces every table and figure of the revised manuscript. It extends the
original v2 notebook with the analyses added during revision:

* a full record audit of the cleaning pipeline (Table 1)
* classical time-series baselines and the XGBoost, LightGBM and CatBoost baselines (Table 4)
* HAC Diebold-Mariano tests with Newey-West variance, Harvey correction, count-weighted daily
  aggregation, Holm multiplicity control and confidence intervals (Table 5)
* accuracy and bias within all six CPCB categories, and warning skill POD/FAR/CSI (Tables 6, 7)
* a component ablation separating stacking, episode weighting and the gate (Table 8)
* the design grid re-evaluated on the test period, by year and by region; an out-of-fold
  stacking protocol; and a cities-withheld experiment (Table 9)
* marginal and Mondrian conformal intervals with category-wise coverage (Table 10)
* an exact Shapley decomposition of the complete gated ensemble (Figs 4, 5)

**Data** are pinned to commit `82b3914` of the public bulletin repository, so the record is
fixed regardless of when the notebook is run.

**Run time** on a free two-core CPU: about 30 minutes for the main run over three horizons,
plus roughly 40 minutes for the full set of revision experiments. Set `HORIZONS = [1]` and
turn off the heavier flags below for a quick pass.


## 0  Configuration

In [ ]:
HORIZONS = [1, 2, 3]          # forecast horizons in days
LAM, TAU_ = 4, 150            # episode weight and regime gate, selected on the 2024 validation year
DATA_COMMIT = "82b39146543847b71a7c9be652f24f0d5a8f2607"

RUN_MAIN        = True        # Section 4.2: all models, all horizons
RUN_TABLES      = True        # Sections 4.2-4.5: accuracy, DM tests, categories, warnings, components
RUN_DESIGN_GRID = True        # Table 3 and the test-period grid in Table 9
RUN_ABLATION    = True        # Table 8, lower block
RUN_ROLLING     = True        # Table 9, rolling origin
RUN_UNSEEN      = True        # Table 9, cities withheld from training
RUN_PROTOCOL_B  = True        # Table 9, out-of-fold stacking protocol  (slowest, ~15 min)
RUN_CONFORMAL   = True        # Table 10
RUN_SHAP        = True        # Figs 4 and 5
RUN_FIGURES     = True

import os
os.makedirs("results", exist_ok=True)

try:                      # display() exists in Jupyter and Colab; fall back elsewhere
    display
except NameError:
    def display(*a):
        for x in a:
            print(x)

## 1  Data download, pinned to a fixed commit

In [ ]:
import os, subprocess
if not os.path.exists("AQI_scraper"):
    subprocess.run(["git", "clone", "-q", "https://github.com/athulvis/AQI_scraper"], check=True)
    subprocess.run(["git", "-C", "AQI_scraper", "checkout", "-q", DATA_COMMIT], check=False)
print("data folder ready:", os.path.exists("AQI_scraper/AQI.csv"))
try:
    import shap
except ImportError:
    subprocess.run(["pip", "install", "-q", "shap"], check=False)
for pkg in ("xgboost", "lightgbm", "catboost"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run(["pip", "install", "-q", pkg], check=False)

## 2  Parsing, cleaning and the record audit (Table 1)

Every exclusion rule is counted, so the numbers in Table 1 of the paper are produced by the same code that builds the dataset.

In [ ]:
"""Parsing, features, learners and evaluation tools."""
import os, re, csv, glob, time, warnings
import numpy as np, pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")

SEED = 42
STUDY_START, TRAIN_END, VAL_END, TEST_END = "2018-01-29", "2023-12-31", "2024-12-31", "2026-10-14"

CATS = {'good': (0, 50), 'satisfactory': (51, 100), 'moderate': (101, 200),
        'poor': (201, 300), 'very poor': (301, 400), 'severe': (401, 500)}

IGP = {"Delhi","Noida","Greater Noida","Ghaziabad","Gurugram","Faridabad","Meerut","Hapur",
"Bulandshahr","Baghpat","Muzaffarnagar","Moradabad","Bareilly","Agra","Firozabad","Mathura",
"Vrindavan","Kanpur","Lucknow","Prayagraj","Varanasi","Gorakhpur","Patna","Gaya","Muzaffarpur",
"Hajipur","Panipat","Sonipat","Rohtak","Bhiwani","Jind","Hisar","Fatehabad","Kaithal","Karnal",
"Kurukshetra","Yamuna Nagar","Ambala","Bahadurgarh","Ballabgarh","Dharuhera","Manesar",
"Charkhi Dadri","Narnaul","Palwal","Amritsar","Ludhiana","Jalandhar","Patiala","Bathinda",
"Khanna","Mandi Gobindgarh","Rupnagar","Chandigarh","Kolkata","Howrah","Asansol","Durgapur",
"Siliguri"}

dpat = re.compile(r"(20\d\d)-?(\d\d)-?(\d\d)$")


def files(root):
    fs = sorted(glob.glob(f"{root}/data/Data_upto_20231218/*.csv")) + \
         [f"{root}/data/20231201_20250304/AQI.csv", f"{root}/AQI.csv"]
    return fs


def parse(root):
    """Returns the cleaned frame plus a full audit trail of every exclusion."""
    rows = []
    audit = dict(raw=0, no_category=0, no_date=0, no_city=0, unparsable=0, out_of_range=0)
    for f in files(root):
        if not os.path.exists(f):
            continue
        with open(f, encoding="utf8", errors="ignore") as fh:
            for r in csv.reader(fh):
                r = [x.strip() for x in r]
                if not any(r) or "Air Quality" in r:
                    if r and r[-1].startswith("Date") and len(r[-1]) > 4:
                        r = [r[-1][4:]] + r
                    else:
                        continue
                audit['raw'] += 1
                cat = next((x.lower() for x in r if x.lower() in CATS), None)
                date = None
                for x in r[::-1]:
                    m = dpat.match(x)
                    if m:
                        date = f"{m[1]}-{m[2]}-{m[3]}"
                        break
                vals = [x for x in r if x]
                city = vals[0] if vals else None
                if r[0] == "" and len(r) > 1:
                    city = r[1]
                try:
                    i = r.index(next(x for x in r if x.lower() == cat))
                    aqi = float(r[i + 1])
                    pol = r[i + 2] if len(r) > i + 2 else ""
                except Exception:
                    audit['unparsable'] += 1
                    continue
                if cat is None:
                    audit['no_category'] += 1; continue
                if date is None:
                    audit['no_date'] += 1; continue
                if not city:
                    audit['no_city'] += 1; continue
                lo, hi = CATS[cat]
                if not (lo <= aqi <= hi):
                    audit['out_of_range'] += 1; continue
                rows.append((city, date, aqi, pol))
    df = pd.DataFrame(rows, columns=["city", "date", "aqi", "pollutant"])
    audit['kept_before_dedup'] = len(df)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    n0 = len(df); df = df.dropna(subset=["date"]); audit['bad_date_dropped'] = n0 - len(df)
    key = df["city"].str.replace(r"\s+", " ", regex=True).str.strip().str.lower()
    canon = df.assign(k=key).groupby("k")["city"].agg(lambda s: s.value_counts().index[0])
    audit['distinct_city_strings'] = int(df['city'].nunique())
    df["city"] = key.map(canon)
    audit['distinct_cities_after_merge'] = int(df['city'].nunique())
    n0 = len(df); df = df.drop_duplicates(["city", "date"], keep="last")
    audit['duplicate_city_days_dropped'] = n0 - len(df)
    audit['valid_city_days'] = len(df)
    return df, audit


def pollutant_flags(s):
    s = s.fillna("").astype(str)
    pm25 = s.str.contains("2.5", regex=False)
    pm10 = s.str.contains("10", regex=False)
    gas = s.str.contains(r"CO|NO|SO|O3|NH", regex=True) & ~(pm25 | pm10)
    return pd.DataFrame({"pol_pm25": pm25.astype(int), "pol_pm10": pm10.astype(int),
                         "pol_gas": gas.astype(int)})


GROUPS = {
 "Recent history": ["lag0","lag1","lag2","lag3","lag4","lag5","lag6","lag13","lag_1y"],
 "Trends": ["roll_mean3","roll_mean7","roll_mean14","roll_mean30","roll_std7","roll_max7","diff1","diff7"],
 "Regional": ["nat_mean","igp_mean","igp_mean_diff1","city_minus_igp"],
 "City context": ["is_igp","city_mean_aqi","pol_pm25","pol_pm10","pol_gas","obs_days7"],
 "Calendar": ["doy_sin","doy_cos","month","weekday","stubble_season"]}
FEATURES = sum(GROUPS.values(), [])


def build_dataset(full, h=1, min_days=2500):
    full = full[full.Date >= STUDY_START]
    counts = full.groupby("City").size(); cities = counts[counts >= min_days].index
    d = full[full.City.isin(cities)].reset_index(drop=True)
    d = pd.concat([d, pollutant_flags(d["Pollutant"])], axis=1)
    nat = full.groupby("Date").AQI.mean().rename("nat_mean")
    igp = full[full.City.isin(IGP)].groupby("Date").AQI.mean().rename("igp_mean")
    reg = pd.concat([nat, igp], axis=1)
    reg = reg.reindex(pd.date_range(reg.index.min(), reg.index.max(), freq="D"))
    out = []; dates = pd.date_range(STUDY_START, d.Date.max(), freq="D")
    for city, g in d.groupby("City"):
        g = g.set_index("Date").reindex(dates); a = g["AQI"]
        f = pd.DataFrame(index=dates); f["City"] = city
        f["target"] = a.shift(-h)
        for k in range(7): f[f"lag{k}"] = a.shift(k)
        f["lag13"] = a.shift(13); f["lag_1y"] = a.shift(365 - h)
        for w in (3, 7, 14, 30):
            f[f"roll_mean{w}"] = a.rolling(w, min_periods=max(2, w // 2)).mean()
        f["roll_std7"] = a.rolling(7, min_periods=4).std()
        f["roll_max7"] = a.rolling(7, min_periods=4).max()
        f["diff1"] = a - a.shift(1); f["diff7"] = a - a.shift(7)
        f["obs_days7"] = a.notna().rolling(7, min_periods=1).sum()
        for c in ("pol_pm25", "pol_pm10", "pol_gas"): f[c] = g[c]
        f["nat_mean"] = reg["nat_mean"].reindex(dates).values
        f["igp_mean"] = reg["igp_mean"].reindex(dates).values
        f["igp_mean_diff1"] = f["igp_mean"] - reg["igp_mean"].shift(1).reindex(dates).values
        f["city_minus_igp"] = f["lag0"] - f["igp_mean"]; f["is_igp"] = int(city in IGP)
        t = dates + pd.Timedelta(days=h); doy = t.dayofyear
        f["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
        f["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
        f["month"] = t.month; f["weekday"] = t.weekday
        f["stubble_season"] = ((t.month == 10) | (t.month == 11)).astype(int)
        out.append(f)
    data = pd.concat(out).rename_axis("Date").reset_index().dropna(subset=["target", "lag0"])
    data["TargetDate"] = data.Date + pd.Timedelta(days=h)
    data = data[data.TargetDate <= TEST_END]
    data["split"] = np.where(data.TargetDate <= TRAIN_END, "train",
                             np.where(data.TargetDate <= VAL_END, "val", "test"))
    enc = data[data.split == "train"].groupby("City").lag0.mean().rename("city_mean_aqi")
    return data.merge(enc, left_on="City", right_index=True, how="left").reset_index(drop=True)


class MedianFill(BaseEstimator, RegressorMixin):
    def __init__(self, est=None): self.est = est
    def fit(self, X, y, sample_weight=None):
        X = pd.DataFrame(X); self.med_ = X.median()
        self.est_ = clone(self.est).fit(X.fillna(self.med_).fillna(0), y, sample_weight=sample_weight)
        return self
    def predict(self, X):
        return self.est_.predict(pd.DataFrame(X).fillna(self.med_).fillna(0))


def make(name):
    if name == "rf":
        return RandomForestRegressor(n_estimators=120, min_samples_leaf=5, max_features=0.5,
                                     max_samples=0.4, n_jobs=-1, random_state=SEED)
    if name == "et":
        return MedianFill(ExtraTreesRegressor(n_estimators=120, min_samples_leaf=5,
                                              max_features=0.5, n_jobs=-1, random_state=SEED))
    if name == "hgb":
        return HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05, max_leaf_nodes=48,
                                             min_samples_leaf=40, l2_regularization=1.0,
                                             random_state=SEED)
    if name == "gbr":
        return MedianFill(GradientBoostingRegressor(n_estimators=220, max_depth=5,
                                                    learning_rate=0.06, subsample=0.4,
                                                    max_features=0.5, random_state=SEED))
    raise KeyError(name)


LABEL = {"rf": "Random Forest", "et": "Extra Trees", "gbr": "Gradient Boosting",
         "hgb": "Hist. Gradient Boosting"}

CAT_NAMES = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]


def aqi_category(a):
    return np.digitize(np.asarray(a), [50.5, 100.5, 200.5, 300.5, 400.5])


def metrics(y, p):
    y, p = np.asarray(y, float), np.clip(np.asarray(p, float), 0, 500)
    cy, cp = aqi_category(y), aqi_category(p)
    r = lambda m: float(np.sqrt(mean_squared_error(y[m], p[m]))) if m.any() else np.nan
    return {"RMSE": r(y >= 0), "MAE": mean_absolute_error(y, p), "R2": r2_score(y, p),
            "MAPE (%)": 100 * np.mean(np.abs(y - p) / np.maximum(y, 1)),
            "Category acc. (%)": 100 * np.mean(cy == cp),
            "Within one category (%)": 100 * np.mean(np.abs(cy - cp) <= 1),
            "RMSE AQI>200": r(y > 200), "RMSE AQI>300": r(y > 300)}


def newey_west_var(x, lag):
    x = np.asarray(x, float) - np.mean(x); T = len(x); v = np.dot(x, x) / T
    for k in range(1, lag + 1):
        v += 2 * (1 - k / (lag + 1)) * np.dot(x[k:], x[:-k]) / T
    return v


def dm_hac(P, a, b, h, subset=None):
    """Diebold-Mariano test on squared-error differences, aggregated by target
    date so that same-day dependence across cities is absorbed, and with a
    Newey-West HAC variance plus Harvey's small-sample correction for the
    MA(h-1) structure of h-step forecast errors.

    Days are weighted by the number of qualifying city-days, so the estimated
    mean loss differential equals the difference in pooled mean squared error
    over the same subset; the test and the reported RMSE therefore always agree
    in sign.  Positive statistic => b better than a.
    """
    Q = P if subset is None else P[subset]
    dif = ((Q.y - Q[a].clip(0, 500)) ** 2 - (Q.y - Q[b].clip(0, 500)) ** 2)
    g = dif.groupby(Q.TargetDate)
    s = g.sum().sort_index()          # daily sum of loss differences
    m = g.size().reindex(s.index)     # city-days contributing that day
    T = len(s)
    mu = float(s.sum() / m.sum())     # = MSE(a) - MSE(b) over the subset
    u = (s.values - mu * m.values) / m.values.mean()
    lag = max(h - 1, int(np.floor(4 * (T / 100) ** (2 / 9))))
    var = newey_west_var(u, lag) / T
    harvey = np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    se = np.sqrt(var) / harvey
    stat = mu / se if se > 0 else np.nan
    tcrit = stats.t.ppf(0.975, df=T - 1)
    return {"DM statistic": float(stat),
            "p-value": float(2 * (1 - stats.t.cdf(abs(stat), df=T - 1))),
            "mean loss diff": float(mu),
            "CI low": float(mu - tcrit * se),
            "CI high": float(mu + tcrit * se),
            "HAC lag": int(lag), "n days": int(T)}


def holm(pvals):
    """Holm-Bonferroni adjusted p-values, order preserved."""
    p = np.asarray(pvals, float); m = len(p); o = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for i, k in enumerate(o):
        run = max(run, (m - i) * p[k]); adj[k] = min(1.0, run)
    return adj


def load_full(root):
    df, audit = parse(root)
    full = df.rename(columns={"city": "City", "date": "Date", "aqi": "AQI",
                              "pollutant": "Pollutant"})
    return full, audit


In [ ]:
full, AUDIT = load_full("AQI_scraper")
import json
json.dump(AUDIT, open("results/audit.json", "w"), indent=1)
study = full[(full.Date >= STUDY_START) & (full.Date <= TEST_END)]
n = study.groupby("City").size()
print("Table 1 - record audit")
for k, v in AUDIT.items():
    print(f"  {k:30s} {v:,}")
print(f"  study-period records            {len(study):,} from {study.City.nunique()} cities")
print(f"  cities with >= 2,500 records    {(n >= 2500).sum()}")
STUDY = full[full.Date <= TEST_END]
D1 = build_dataset(STUDY, 1)
D1.to_pickle("results/D1.pkl")
print(D1.split.value_counts().to_string(), "| total", len(D1))

## 3  The main run: RASE, the baselines and the three horizons

Adds three classical time-series baselines and the XGBoost, LightGBM and CatBoost comparators that the reviewers asked for.

In [ ]:
"""Stage A - the main run: all learners, all baselines, three horizons.
Writes results/predictions_h{h}.csv.gz with one column per model.
"""
import sys, os, time, json
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

OUT = 'results'
RES = OUT
os.makedirs(RES, exist_ok=True)
LAM, TAU_ = 4, 150


def fit_stream(X, d, w, tr, va, te, tag="", names=("rf", "et", "gbr", "hgb")):
    trva = tr | va
    V, T = {}, {}
    for n in names:
        s = time.time()
        V[n] = make(n).fit(X[tr], d[tr], sample_weight=None if w is None else w[tr]).predict(X[va])
        T[n] = make(n).fit(X[trva], d[trva], sample_weight=None if w is None else w[trva]).predict(X[te])
        print(f"   [{tag}] {LABEL[n]:24s} {time.time()-s:6.0f} s", flush=True)
    meta = LinearRegression(positive=True).fit(np.column_stack([V[n] for n in names]), d[va],
                                               sample_weight=None if w is None else w[va])
    Tm = np.column_stack([T[n] for n in names])
    return {"single": T, "stack": meta.predict(Tm), "avg": Tm.mean(1),
            "weights": dict(zip(names, meta.coef_.round(4))),
            "intercept": float(meta.intercept_)}


# ---------------------------------------------------------------- new baselines
def gbdt_baselines(X, d, tr, va, te):
    """XGBoost, LightGBM and CatBoost on the same change target."""
    import xgboost as xgb, lightgbm as lgb
    from catboost import CatBoostRegressor
    trva = tr | va
    out = {}
    Xf = X.fillna(X.median()).fillna(0)
    t = time.time()
    m = xgb.XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                         colsample_bytree=0.8, reg_lambda=1.0, n_jobs=2, random_state=SEED,
                         tree_method="hist")
    m.fit(X[trva], d[trva]); out["XGBoost"] = m.predict(X[te]); print(f"   XGBoost {time.time()-t:.0f}s", flush=True)
    t = time.time()
    m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=48, min_child_samples=40,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                          n_jobs=2, random_state=SEED, verbose=-1)
    m.fit(X[trva], d[trva]); out["LightGBM"] = m.predict(X[te]); print(f"   LightGBM {time.time()-t:.0f}s", flush=True)
    t = time.time()
    m = CatBoostRegressor(iterations=600, learning_rate=0.05, depth=6, l2_leaf_reg=3.0,
                          random_seed=SEED, verbose=0, thread_count=2)
    m.fit(Xf[trva], d[trva]); out["CatBoost"] = m.predict(Xf[te]); print(f"   CatBoost {time.time()-t:.0f}s", flush=True)
    return out


def ts_baselines(data, h, tr, va, te):
    """Classical time-series baselines, computed per city.
    climatology: city x day-of-year mean over the training span.
    holt       : damped-trend exponential smoothing, h-step ahead at every origin.
    ar6        : AR(6) on AQI levels fitted per city on train+val, iterated h steps.
    All use information available at the forecast issue date only.
    """
    trva = (tr | va)
    out = {k: np.full(te.sum(), np.nan) for k in ("Climatology", "Exponential smoothing", "AR(6)")}
    te_idx = np.where(te)[0]
    pos = {ix: j for j, ix in enumerate(te_idx)}
    d_tr = data.loc[trva]
    # climatology: mean AQI by city and day-of-year of the TARGET date, training span only
    doy_tr = d_tr.TargetDate.dt.dayofyear
    clim = d_tr.assign(doy=doy_tr).groupby(["City", "doy"]).target.mean()
    city_mean = d_tr.groupby("City").target.mean()
    sub = data.loc[te]
    key = list(zip(sub.City, sub.TargetDate.dt.dayofyear))
    cvals = clim.reindex(pd.MultiIndex.from_tuples(key)).values
    fallback = sub.City.map(city_mean).values
    out["Climatology"] = np.where(np.isnan(cvals), fallback, cvals)

    # Holt damped trend and AR(6), per city, on the contiguous daily AQI series
    lagcols = [f"lag{k}" for k in range(7)]
    for city, g in data.groupby("City"):
        g = g.sort_values("Date")
        gi = g.index.values
        is_te = te[gi]
        # --- Holt damped trend on lag0 history, recursion over this city's dates
        a = g.lag0.values.astype(float)
        alpha, beta, phi = 0.4, 0.1, 0.9
        lvl = a[0]; trd = 0.0
        hf = np.empty(len(a))
        for i in range(len(a)):
            # forecast h steps ahead from the state BEFORE seeing day i+1
            damp = sum(phi ** (k + 1) for k in range(h))
            hf[i] = lvl + damp * trd
            if not np.isnan(a[i]):
                prev = lvl
                lvl = alpha * a[i] + (1 - alpha) * (lvl + phi * trd)
                trd = beta * (lvl - prev) + (1 - beta) * phi * trd
        m = is_te
        if m.any():
            for ix, v in zip(gi[m], hf[m]):
                out["Exponential smoothing"][pos[ix]] = v
        # --- AR(6) on levels, OLS on train+val rows of this city, iterated h steps
        gm = trva[gi]
        Z = g.loc[:, lagcols].values.astype(float)
        yv = g.lag0.values.astype(float)
        ok = gm & ~np.isnan(Z).any(1) & ~np.isnan(yv)
        if ok.sum() > 200:
            # regress AQI(t) on AQI(t-1) ... AQI(t-6), i.e. lag1..lag6 predicting lag0
            Zt = Z[ok]
            A = np.column_stack([np.ones(ok.sum()), Zt[:, 1], Zt[:, 2], Zt[:, 3], Zt[:, 4], Zt[:, 5], Zt[:, 6]])
            b, *_ = np.linalg.lstsq(A, yv[ok], rcond=None)
            # iterate: state = [lag0..lag5] -> predict next
            Zte = Z[is_te]
            hist = [Zte[:, k] for k in range(6)]  # lag0..lag5 at issue time
            cur = None
            for step in range(h):
                xs = np.column_stack([np.ones(len(Zte))] + hist[:6])
                cur = xs @ b
                hist = [cur] + hist[:5]
            vals = cur
        else:
            vals = np.full(int(is_te.sum()), np.nan)
        for ix, v in zip(gi[is_te], vals):
            out["AR(6)"][pos[ix]] = v
    # any remaining gaps fall back to persistence
    pers = data.loc[te, "lag0"].values.astype(float)
    for k in out:
        v = np.asarray(out[k], float)
        out[k] = np.where(np.isnan(v), pers, v)
    return out


def run_horizon(full, h):
    print(f"\n===== Horizon h = {h} day(s) =====", flush=True)
    data = build_dataset(full, h)
    X = data[FEATURES].astype(float)
    y = data.target.values.astype(float)
    l0 = data.lag0.values.astype(float)
    d = y - l0
    tr, va, te = [(data.split == s).values for s in ("train", "val", "test")]
    trva = tr | va
    print(f" train {tr.sum():,} | validation {va.sum():,} | test {te.sum():,}", flush=True)
    S = fit_stream(X, d, None, tr, va, te, "standard")
    print("   standard-stack weights:", S["weights"], flush=True)
    E = fit_stream(X, d, 1.0 + LAM * (y > 300), tr, va, te, "episode")
    print("   episode-stack weights:", E["weights"], flush=True)
    ridge = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          Ridge(alpha=1.0)).fit(X[trva], d[trva])
    mlp = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                        MLPRegressor(hidden_layer_sizes=(128, 64), early_stopping=True,
                                     max_iter=200, random_state=SEED)).fit(X[trva], d[trva])
    a = l0[te]
    P = data.loc[te, ["Date", "TargetDate", "City"]].reset_index(drop=True)
    P["y"] = y[te]
    P["Persistence"] = a
    for k, v in ts_baselines(data, h, tr, va, te).items():
        P[k] = v
    P["Ridge regression"] = a + ridge.predict(X[te])
    P["Multilayer perceptron"] = a + mlp.predict(X[te])
    for k, v in S["single"].items():
        P[LABEL[k]] = a + v
    for k, v in gbdt_baselines(X, d, tr, va, te).items():
        P[k] = a + v
    P["Average ensemble"] = a + S["avg"]
    P["Standard stack"] = a + S["stack"]
    P["Episode stack"] = a + E["stack"]
    P["RASE (proposed)"] = np.where(a >= TAU_, P["Episode stack"], P["Standard stack"])
    P.to_csv(f"{RES}/predictions_h{h}.csv.gz", index=False)
    json.dump({"standard_weights": {k: float(v) for k, v in S["weights"].items()},
               "episode_weights": {k: float(v) for k, v in E["weights"].items()},
               "standard_intercept": S["intercept"], "episode_intercept": E["intercept"],
               "gate_share_pct": float(100 * np.mean(a >= TAU_)),
               "n_train": int(tr.sum()), "n_val": int(va.sum()), "n_test": int(te.sum()),
               "n_total": int(len(data))},
              open(f"{RES}/run_h{h}.json", "w"), indent=1)
    print(f" gate routed {100*np.mean(a >= TAU_):.1f}% of test days to the episode stream", flush=True)
    del data, X
    return P


In [ ]:
PRED = {}
if RUN_MAIN:
    for h in HORIZONS:
        PRED[h] = run_horizon(STUDY, h)
else:
    for h in HORIZONS:
        PRED[h] = pd.read_csv(f'results/predictions_h{h}.csv.gz',
                              parse_dates=['Date', 'TargetDate'])

## 4  Tables 4 to 8: accuracy, significance, categories, warnings, components

In [ ]:
"""Stage B - every table that follows from the saved predictions.
No model fitting here, so each number traces back to one run.
"""
import sys, os, json
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

OUT = 'results'
RES = OUT
TAB = OUT
os.makedirs(TAB, exist_ok=True)
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, np.clip(b, 0, 500))))

ORDER = ["Persistence", "Climatology", "Exponential smoothing", "AR(6)", "Ridge regression",
         "Multilayer perceptron", "Random Forest", "Extra Trees", "Gradient Boosting",
         "Hist. Gradient Boosting", "XGBoost", "LightGBM", "CatBoost", "Average ensemble",
         "Standard stack", "Episode stack", "RASE (proposed)"]


def load(h):
    P = pd.read_csv(f"{RES}/predictions_h{h}.csv.gz", parse_dates=["Date", "TargetDate"])
    return P


def b1_main(Ps):
    rows = []
    for h, P in Ps.items():
        for m in ORDER:
            if m not in P.columns:
                continue
            d = metrics(P.y, P[m])
            d.update({"Horizon": h, "Model": m})
            rows.append(d)
    T = pd.DataFrame(rows).set_index(["Horizon", "Model"])
    T.round(3).to_csv(f"{TAB}/b1_main_multihorizon.csv")
    return T


def b2_categories(Ps, models=("Persistence", "Standard stack", "CatBoost", "RASE (proposed)")):
    rows = []
    for h, P in Ps.items():
        cy = aqi_category(P.y.values)
        for b, nm in enumerate(CAT_NAMES):
            m = cy == b
            if m.sum() == 0:
                continue
            rec = {"Horizon": h, "Category": nm, "n": int(m.sum())}
            for mod in models:
                if mod not in P.columns:
                    continue
                p = np.clip(P[mod].values, 0, 500)
                rec[f"{mod} RMSE"] = rmse(P.y.values[m], p[m])
                rec[f"{mod} MAE"] = float(mean_absolute_error(P.y.values[m], p[m]))
                rec[f"{mod} bias"] = float(np.mean(p[m] - P.y.values[m]))
            rows.append(rec)
    T = pd.DataFrame(rows)
    T.round(2).to_csv(f"{TAB}/b2_categories.csv", index=False)
    return T


def b3_dm(Ps):
    rows = []
    for h, P in Ps.items():
        models = [m for m in ORDER if m in P.columns and m != "RASE (proposed)"]
        for subset, nm in ((None, "All days"), ((P.y > 200).values, "AQI > 200"),
                           ((P.y > 300).values, "AQI > 300")):
            recs = []
            for m in models:
                r = dm_hac(P, m, "RASE (proposed)", h, subset=subset)
                r.update({"Horizon": h, "Subset": nm, "Comparison": f"{m} vs RASE"})
                recs.append(r)
            p = [r["p-value"] for r in recs]
            adj = holm(p)
            for r, a in zip(recs, adj):
                r["p (Holm)"] = float(a)
                rows.append(r)
    T = pd.DataFrame(rows)[["Horizon", "Subset", "Comparison", "DM statistic", "p-value",
                            "p (Holm)", "mean loss diff", "CI low", "CI high", "HAC lag", "n days"]]
    T.round(4).to_csv(f"{TAB}/b3_dm_tests.csv", index=False)
    return T


def b4_cities(Ps):
    P = Ps[1]
    rows = []
    singles = [m for m in ("Random Forest", "Extra Trees", "Gradient Boosting",
                           "Hist. Gradient Boosting") if m in P.columns]
    for c, g in P.groupby("City"):
        rows.append({"City": c, "Test days": len(g),
                     "Mean AQI": float(g.y.mean()),
                     "Days AQI>300": int((g.y > 300).sum()),
                     "Persistence": rmse(g.y, g["Persistence"]),
                     "Best single learner": min(rmse(g.y, g[m]) for m in singles),
                     "Standard stack": rmse(g.y, g["Standard stack"]),
                     "RASE": rmse(g.y, g["RASE (proposed)"]),
                     "RASE R2": float(r2_score(g.y, np.clip(g["RASE (proposed)"], 0, 500)))})
    S = pd.DataFrame(rows).set_index("City")
    S["Reduction vs persistence (%)"] = 100 * (1 - S.RASE / S.Persistence)
    S.round(3).to_csv(f"{TAB}/b4_cities.csv")
    return S


def b5_warnings(Ps, thresholds=(200, 300)):
    """Contingency skill for issuing a warning: the forecast crosses the
    threshold when the observation does."""
    rows = []
    for h, P in Ps.items():
        for thr in thresholds:
            obs = (P.y.values > thr)
            for m in ORDER:
                if m not in P.columns:
                    continue
                fc = (np.clip(P[m].values, 0, 500) > thr)
                H = int((fc & obs).sum()); M = int((~fc & obs).sum())
                F = int((fc & ~obs).sum())
                pod = H / (H + M) if H + M else np.nan
                far = F / (H + F) if H + F else np.nan
                csi = H / (H + M + F) if H + M + F else np.nan
                rows.append({"Horizon": h, "Threshold": thr, "Model": m, "Hits": H,
                             "Misses": M, "False alarms": F, "POD": pod, "FAR": far, "CSI": csi})
    T = pd.DataFrame(rows)
    T.round(3).to_csv(f"{TAB}/b5_warning_skill.csv", index=False)
    return T


def b6_components(Ps):
    """Isolates the three ingredients of RASE from the saved columns."""
    rows = []
    for h, P in Ps.items():
        a = P["Persistence"].values.astype(float)
        singles = ["Random Forest", "Extra Trees", "Gradient Boosting", "Hist. Gradient Boosting"]
        best = min(singles, key=lambda m: rmse(P.y, P[m]))
        comp = {"Persistence": a,
                f"Change target, best single learner ({best})": P[best].values,
                "Stacking only (standard stack)": P["Standard stack"].values,
                "Episode weighting only (episode stack)": P["Episode stack"].values,
                "Stacking + weighting + gate (RASE)": P["RASE (proposed)"].values}
        for nm, p in comp.items():
            y = P.y.values
            rows.append({"Horizon": h, "Configuration": nm, "RMSE": rmse(y, p),
                         "MAE": float(mean_absolute_error(y, np.clip(p, 0, 500))),
                         "RMSE >200": rmse(y[y > 200], p[y > 200]),
                         "RMSE >300": rmse(y[y > 300], p[y > 300])})
    T = pd.DataFrame(rows)
    T.round(2).to_csv(f"{TAB}/b6_components.csv", index=False)
    return T


def b7_protocols(Ps):
    """Protocol A (published) against Protocol B (out-of-fold meta-learner,
    design year separated from the calibration year)."""
    f = f"{RES}/c5_protocol_b_meta.json"
    if not os.path.exists(f):
        return None
    meta = json.load(open(f))
    P = Ps[1]
    names = ["Random Forest", "Extra Trees", "Gradient Boosting", "Hist. Gradient Boosting"]
    a = P["Persistence"].values.astype(float)
    base_s = np.column_stack([P[n].values - a for n in names])
    epi_file = f"{TAB}/epi_base_test.npy"
    if not os.path.exists(epi_file):
        print("episode-stream base predictions missing; skipping protocol comparison")
        return None
    base_e = np.load(epi_file)
    cS, iS = np.asarray(meta["std"]["coef"]), meta["std"]["intercept"]
    cE, iE = np.asarray(meta["epi"]["coef"]), meta["epi"]["intercept"]
    pS = a + base_s @ cS + iS
    pE = a + base_e @ cE + iE
    tau = meta.get("tau_selected_on_2023", 150)
    pB = np.where(a >= tau, pE, pS)
    y = P.y.values
    rows = []
    for nm, p in (("Protocol A: meta on 2024, tau = 150 (as published)", P["RASE (proposed)"].values),
                  (f"Protocol B: out-of-fold meta, tau = {tau} chosen on 2023", pB)):
        rows.append({"Protocol": nm, "RMSE": rmse(y, p),
                     "MAE": float(mean_absolute_error(y, np.clip(p, 0, 500))),
                     "R2": float(r2_score(y, np.clip(p, 0, 500))),
                     "RMSE >200": rmse(y[y > 200], p[y > 200]),
                     "RMSE >300": rmse(y[y > 300], p[y > 300])})
    T = pd.DataFrame(rows)
    T.round(3).to_csv(f"{TAB}/b7_protocols.csv", index=False)
    # DM between the two protocols on severe days
    Q = P.copy(); Q["Protocol B"] = pB
    r = dm_hac(Q, "Protocol B", "RASE (proposed)", 1, subset=(Q.y > 300).values)
    json.dump(r, open(f"{TAB}/b7_protocol_dm.json", "w"), indent=1)
    return T


In [ ]:
if RUN_TABLES:
    Ps = PRED
    T = b1_main(Ps); display(T.loc[HORIZONS[0]].round(2))
    display(b2_categories(Ps).round(2))
    display(b3_dm(Ps).head(20).round(4))
    display(b5_warnings(Ps).round(3))
    display(b6_components(Ps).round(2))
    b4_cities(Ps)

## 5  Tables 3 and 9: design grid, ablation, rolling origin, protocol, unseen cities

The design grid is evaluated both on the 2024 validation year, where the configuration is chosen, and on the untouched test period, where its robustness is measured.

In [ ]:
"""Stage C - the experiments the reviewers asked for that need model fitting.

C1  design-selection grid on the 2024 validation year (full lambda x tau grid)
C2  feature-group ablation on the test period (proxy learner)
C3  rolling-origin evaluation (proxy learner)
C4  lambda/tau sensitivity measured on the TEST period, by year and by region
C5  Protocol B: chronological out-of-fold stacking, design year separated from
    the calibration year  (reviewer 2, point 3)
C6  conformal prediction: marginal and Mondrian (regime-conditional) intervals
    with category-wise coverage and width  (reviewer 2, point 7)
C7  unseen-city generalisation: grouped 5-fold over cities  (reviewer 2, point 8)
"""
import sys, os, time, json
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

OUT = 'results'
RES = OUT
os.makedirs(RES, exist_ok=True)
LAM, TAU_ = 4, 150
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, np.clip(b, 0, 500))))


def log(*a):
    print(*a, flush=True)


# ------------------------------------------------------------------ C1 + C4
def c1_c4(D1):
    X = D1[FEATURES].astype(float)
    y = D1.target.values.astype(float)
    l0 = D1.lag0.values.astype(float)
    d = y - l0
    tr, va, te = [(D1.split == s).values for s in ("train", "val", "test")]
    trva = tr | va
    LAMS = [0, 1, 2, 4, 6, 8]
    TAUS = [0, 100, 150, 200, 250, 300]

    # --- C1: fitted on train, evaluated on the 2024 validation year
    yv, av = y[va], l0[va]
    dS_v = make("hgb").fit(X[tr], d[tr]).predict(X[va])
    lvl_v = make("hgb").fit(X[tr], y[tr]).predict(X[va])
    dE_v = {}
    for lam in LAMS:
        if lam == 0:
            dE_v[lam] = dS_v
        else:
            dE_v[lam] = make("hgb").fit(X[tr], d[tr], sample_weight=1.0 + lam * (y[tr] > 300)).predict(X[va])
        log(f"   C1 lambda={lam} done")
    rows = {"Persistence": av, "AQI-level target": lvl_v, "AQI-change target": av + dS_v}
    for lam in LAMS:
        if lam:
            rows[f"Change target, episode weight lambda = {lam}"] = av + dE_v[lam]
    for tau in [100, 150, 200, 250]:
        rows[f"Regime gate tau = {tau}, lambda = 4"] = av + np.where(av >= tau, dE_v[4], dS_v)
    r = lambda p, m: rmse(yv[m], p[m])
    T1 = pd.DataFrame({k: [r(p, yv >= 0), r(p, yv > 200), r(p, yv > 300)] for k, p in rows.items()},
                      index=["RMSE (all days)", "RMSE (AQI > 200)", "RMSE (AQI > 300)"]).T.round(2)
    T1.to_csv(f"{RES}/c1_table1_validation.csv")

    # full grid on validation
    grid_v = []
    for lam in LAMS:
        for tau in TAUS:
            p = av + np.where(av >= tau, dE_v[lam], dS_v)
            grid_v.append({"lambda": lam, "tau": tau, "RMSE all": r(p, yv >= 0),
                           "RMSE >200": r(p, yv > 200), "RMSE >300": r(p, yv > 300)})
    pd.DataFrame(grid_v).to_csv(f"{RES}/c4_grid_validation.csv", index=False)

    # --- C4: the same grid, now measured on the untouched test period
    yt, at = y[te], l0[te]
    dS_t = make("hgb").fit(X[trva], d[trva]).predict(X[te])
    dE_t = {}
    for lam in LAMS:
        if lam == 0:
            dE_t[lam] = dS_t
        else:
            dE_t[lam] = make("hgb").fit(X[trva], d[trva], sample_weight=1.0 + lam * (y[trva] > 300)).predict(X[te])
        log(f"   C4 lambda={lam} done")
    sub = D1.loc[te].reset_index(drop=True)
    yr = sub.TargetDate.dt.year.values
    igp = sub.City.isin(IGP).values
    rows_t = []
    for lam in LAMS:
        for tau in TAUS:
            p = at + np.where(at >= tau, dE_t[lam], dS_t)
            rows_t.append({"lambda": lam, "tau": tau,
                           "RMSE all": rmse(yt, p), "RMSE >200": rmse(yt[yt > 200], p[yt > 200]),
                           "RMSE >300": rmse(yt[yt > 300], p[yt > 300]),
                           "RMSE >300 2025": rmse(yt[(yt > 300) & (yr == 2025)], p[(yt > 300) & (yr == 2025)]),
                           "RMSE >300 2026": rmse(yt[(yt > 300) & (yr == 2026)], p[(yt > 300) & (yr == 2026)]),
                           "RMSE >300 IGP": rmse(yt[(yt > 300) & igp], p[(yt > 300) & igp]),
                           "RMSE >300 non-IGP": rmse(yt[(yt > 300) & ~igp], p[(yt > 300) & ~igp])
                           if ((yt > 300) & ~igp).sum() > 5 else np.nan})
    pd.DataFrame(rows_t).round(2).to_csv(f"{RES}/c4_grid_test.csv", index=False)
    log("C1/C4 done")


# ------------------------------------------------------------------ C2 + C3
def c2_c3(D1):
    X = D1[FEATURES].astype(float)
    y = D1.target.values
    l0 = D1.lag0.values
    d = y - l0
    trva, te = (D1.split != "test").values, (D1.split == "test").values

    def proxy(fs):
        return metrics(y[te], l0[te] + make("hgb").fit(X.loc[trva, fs], d[trva]).predict(X.loc[te, fs]))

    sets = {"All features": FEATURES,
            **{f"Without {g.lower()}": [f for f in FEATURES if f not in v] for g, v in GROUPS.items()},
            "Recent history only": GROUPS["Recent history"]}
    T4 = pd.DataFrame({k: proxy(v) for k, v in sets.items()}).T[["RMSE", "MAE", "RMSE AQI>200", "RMSE AQI>300"]]
    T4["Change in RMSE (%)"] = 100 * (T4.RMSE / T4.loc["All features", "RMSE"] - 1)
    T4.round(2).to_csv(f"{RES}/c2_ablation.csv")
    log("C2 done")

    out = {}
    for name, (s, e) in {"2022": ("2022-01-01", "2022-12-31"), "2023": ("2023-01-01", "2023-12-31"),
                         "2024": ("2024-01-01", "2024-12-31"),
                         "2025-26": ("2025-01-01", TEST_END)}.items():
        D = D1.drop(columns="city_mean_aqi")
        trn = D.TargetDate < s
        tst = (D.TargetDate >= s) & (D.TargetDate <= e)
        D = D.merge(D[trn].groupby("City").lag0.mean().rename("city_mean_aqi"),
                    left_on="City", right_index=True, how="left")
        Xr = D[FEATURES].astype(float)
        dd = (D.target - D.lag0).values
        p = np.clip(D.lag0.values[tst.values] + make("hgb").fit(Xr[trn.values], dd[trn.values]).predict(Xr[tst.values]), 0, 500)
        o = D.target.values[tst.values]
        out[name] = {"n test days": int(tst.sum()),
                     "RMSE gain vs persistence (%)": 100 * (1 - np.sqrt(mean_squared_error(o, p)) /
                                                            np.sqrt(mean_squared_error(o, D.lag0.values[tst.values])))}
    pd.DataFrame(out).T.round(1).to_csv(f"{RES}/c3_rolling_origin.csv")
    log("C3 done")


# ------------------------------------------------------------------ C5
def c5_protocol_b(D1, h=1):
    """Chronological out-of-fold stacking.  Base learners are fitted on all
    earlier years only and predict the next year; the meta-learner is fitted on
    those out-of-fold predictions.  lambda and tau are selected on 2023 alone,
    leaving 2024 free for conformal calibration."""
    X = D1[FEATURES].astype(float)
    y = D1.target.values.astype(float)
    l0 = D1.lag0.values.astype(float)
    d = y - l0
    yr = D1.TargetDate.dt.year.values
    te = (D1.split == "test").values
    names = ("rf", "et", "gbr", "hgb")
    # the first fold needs two full years of history so that the one-year lag
    # exists in the fitting set; the 2019 fold is therefore not used
    oof_years = [2020, 2021, 2022, 2023, 2024]
    oof_mask = np.isin(yr, oof_years) & ~te
    OOF = {w: {n: np.full(len(D1), np.nan) for n in names} for w in ("std", "epi")}
    for yy in oof_years:
        fit = (yr < yy) & ~te
        pred = (yr == yy) & ~te
        if fit.sum() < 5000 or pred.sum() == 0:
            continue
        for n in names:
            t = time.time()
            OOF["std"][n][pred] = make(n).fit(X[fit], d[fit]).predict(X[pred])
            w = 1.0 + LAM * (y[fit] > 300)
            OOF["epi"][n][pred] = make(n).fit(X[fit], d[fit], sample_weight=w).predict(X[pred])
            log(f"   C5 OOF {yy} {n} {time.time()-t:.0f}s")
    res = {}
    for w in ("std", "epi"):
        ok = oof_mask & ~np.isnan(OOF[w]["rf"])
        M = np.column_stack([OOF[w][n][ok] for n in names])
        sw = None if w == "std" else 1.0 + LAM * (y[ok] > 300)
        meta = LinearRegression(positive=True).fit(M, d[ok], sample_weight=sw)
        res[w] = {"coef": meta.coef_.tolist(), "intercept": float(meta.intercept_),
                  "n_oof": int(ok.sum())}
    # tau selected on 2023 out-of-fold predictions only
    sel = (yr == 2023) & ~te & ~np.isnan(OOF["std"]["rf"])
    mS = np.column_stack([OOF["std"][n][sel] for n in names])
    mE = np.column_stack([OOF["epi"][n][sel] for n in names])
    pS = np.asarray(res["std"]["coef"]) @ mS.T + res["std"]["intercept"]
    pE = np.asarray(res["epi"]["coef"]) @ mE.T + res["epi"]["intercept"]
    best = None
    for tau in [0, 100, 150, 200, 250, 300]:
        p = l0[sel] + np.where(l0[sel] >= tau, pE, pS)
        sev = y[sel] > 300
        sc = rmse(y[sel][sev], p[sev]) if sev.sum() > 20 else np.inf
        cap = rmse(y[sel], p) <= 1.02 * rmse(y[sel], l0[sel] + pS)
        if cap and (best is None or sc < best[1]):
            best = (tau, sc)
    tau_b = best[0] if best else TAU_
    res["tau_selected_on_2023"] = int(tau_b)
    json.dump(res, open(f"{RES}/c5_protocol_b_meta.json", "w"), indent=1)
    np.save(f"{RES}/c5_oof_std.npy", np.column_stack([OOF['std'][n] for n in names]))
    np.save(f"{RES}/c5_oof_epi.npy", np.column_stack([OOF['epi'][n] for n in names]))
    log("C5 done, tau selected on 2023 =", tau_b)


# ------------------------------------------------------------------ C6
def c6_conformal(D1):
    """Marginal and Mondrian conformalized quantile regression.
    The Mondrian version conditions on the CPCB category of the issue day,
    which is known when the forecast is made, so validity is preserved."""
    X = D1[FEATURES].astype(float)
    y = D1.target.values.astype(float)
    l0 = D1.lag0.values.astype(float)
    d = y - l0
    tr, va, te = [(D1.split == s).values for s in ("train", "val", "test")]
    q = {}
    for al in (0.1, 0.9):
        t = time.time()
        q[al] = HistGradientBoostingRegressor(loss="quantile", quantile=al, max_iter=600,
                                              learning_rate=0.05, max_leaf_nodes=48,
                                              min_samples_leaf=40, random_state=SEED).fit(X[tr], d[tr])
        log(f"   C6 quantile {al} {time.time()-t:.0f}s")
    lo = lambda m: l0[m] + q[0.1].predict(X[m])
    hi = lambda m: l0[m] + q[0.9].predict(X[m])
    lov, hiv, yv = lo(va), hi(va), y[va]
    lot, hit, yt = lo(te), hi(te), y[te]
    sc_v = np.maximum(lov - yv, yv - hiv)

    def conf_q(s):
        n = len(s)
        return np.quantile(s, min(1.0, np.ceil((n + 1) * 0.8) / n)) if n > 10 else np.nan

    Qm = conf_q(sc_v)
    # Mondrian bins on the issue-day category (lag0)
    bin_v, bin_t = aqi_category(l0[va]), aqi_category(l0[te])
    Qb = {}
    for b in range(6):
        m = bin_v == b
        Qb[b] = conf_q(sc_v[m]) if m.sum() > 50 else Qm
    Qt_m = np.full(len(yt), Qm)
    Qt_b = np.array([Qb.get(b, Qm) for b in bin_t])

    # third variant: condition on the predicted category.  The point forecast is
    # a function of information available when the forecast is issued, so this
    # conditioning is admissible and targets the regime the forecast implies
    # rather than the one the issue day was in.
    cen_v = 0.5 * (lov + hiv)
    cen_t = 0.5 * (lot + hit)
    pb_v, pb_t = aqi_category(cen_v), aqi_category(cen_t)
    Qp = {}
    for b in range(6):
        m = pb_v == b
        Qp[b] = conf_q(sc_v[m]) if m.sum() > 50 else Qm
    Qt_p = np.array([Qp.get(b, Qm) for b in pb_t])

    rows = []
    cy = aqi_category(yt)
    for nm, Qt in (("Marginal", Qt_m),
                   ("Mondrian (by issue-day category)", Qt_b),
                   ("Mondrian (by predicted category)", Qt_p)):
        L, U = lot - Qt, hit + Qt
        cov = (yt >= L) & (yt <= U)
        rows.append({"Method": nm, "Subset": "All test days", "n": len(yt),
                     "Coverage (%)": 100 * cov.mean(), "Mean width": float((U - L).mean())})
        for b, nmc in enumerate(CAT_NAMES):
            m = cy == b
            if m.sum() >= 20:
                rows.append({"Method": nm, "Subset": nmc, "n": int(m.sum()),
                             "Coverage (%)": 100 * cov[m].mean(), "Mean width": float((U - L)[m].mean())})
        for thr in (200, 300):
            m = yt > thr
            rows.append({"Method": nm, "Subset": f"AQI > {thr}", "n": int(m.sum()),
                         "Coverage (%)": 100 * cov[m].mean(), "Mean width": float((U - L)[m].mean())})
    unc = (yt >= lot) & (yt <= hit)
    rows.append({"Method": "Uncalibrated quantiles", "Subset": "All test days", "n": len(yt),
                 "Coverage (%)": 100 * unc.mean(), "Mean width": float((hit - lot).mean())})
    pd.DataFrame(rows).round(2).to_csv(f"{RES}/c6_conformal.csv", index=False)
    json.dump({"Q_marginal": float(Qm), "Q_by_issue_bin": {CAT_NAMES[k]: float(v) for k, v in Qb.items()}, "Q_by_pred_bin": {CAT_NAMES[k]: float(v) for k, v in Qp.items()}},
              open(f"{RES}/c6_conformal_q.json", "w"), indent=1)
    log("C6 done")


# ------------------------------------------------------------------ C7
def c7_unseen_cities(D1, k=5):
    """Grouped k-fold over cities: every test city is absent from training.
    city_mean_aqi is replaced by the mean over training cities, because an
    unseen city has no history in the training set."""
    X = D1[FEATURES].astype(float).copy()
    y = D1.target.values.astype(float)
    l0 = D1.lag0.values.astype(float)
    d = y - l0
    te = (D1.split == "test").values
    cities = np.array(sorted(D1.City.unique()))
    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(cities))
    folds = [cities[order[i::k]] for i in range(k)]
    rows = []
    preds = np.full(len(D1), np.nan)
    for i, fold in enumerate(folds):
        inf = D1.City.isin(fold).values
        fit = (~inf) & (~te)
        prd = inf & te
        Xf = X.copy()
        m_city = X.loc[fit, "city_mean_aqi"].mean()
        Xf.loc[inf, "city_mean_aqi"] = m_city
        pS = make("hgb").fit(Xf[fit], d[fit]).predict(Xf[prd])
        pE = make("hgb").fit(Xf[fit], d[fit], sample_weight=1.0 + LAM * (y[fit] > 300)).predict(Xf[prd])
        p = l0[prd] + np.where(l0[prd] >= TAU_, pE, pS)
        preds[prd] = p
        yy = y[prd]
        rows.append({"Fold": i + 1, "Cities": len(fold), "Test days": int(prd.sum()),
                     "Persistence RMSE": rmse(yy, l0[prd]), "RASE RMSE": rmse(yy, p),
                     "Gain (%)": 100 * (1 - rmse(yy, p) / rmse(yy, l0[prd])),
                     "RMSE >300": rmse(yy[yy > 300], p[yy > 300]) if (yy > 300).sum() > 5 else np.nan,
                     "Persistence RMSE >300": rmse(yy[yy > 300], l0[prd][yy > 300]) if (yy > 300).sum() > 5 else np.nan})
        log(f"   C7 fold {i+1} done")
    df = pd.DataFrame(rows)
    yy, pp, aa = y[te], preds[te], l0[te]
    ok = ~np.isnan(pp)
    df.loc[len(df)] = {"Fold": "Pooled", "Cities": len(cities), "Test days": int(ok.sum()),
                       "Persistence RMSE": rmse(yy[ok], aa[ok]), "RASE RMSE": rmse(yy[ok], pp[ok]),
                       "Gain (%)": 100 * (1 - rmse(yy[ok], pp[ok]) / rmse(yy[ok], aa[ok])),
                       "RMSE >300": rmse(yy[ok & (yy > 300)], pp[ok & (yy > 300)]),
                       "Persistence RMSE >300": rmse(yy[ok & (yy > 300)], aa[ok & (yy > 300)])}
    df.round(2).to_csv(f"{RES}/c7_unseen_cities.csv", index=False)
    log("C7 done")


In [ ]:
D1 = pd.read_pickle('results/D1.pkl')
if RUN_ABLATION or RUN_ROLLING: c2_c3(D1)
if RUN_UNSEEN:                  c7_unseen_cities(D1)
if RUN_CONFORMAL:               c6_conformal(D1)
if RUN_DESIGN_GRID:             c1_c4(D1)
if RUN_PROTOCOL_B:              c5_protocol_b(D1)
if RUN_TABLES:                  display(b7_protocols(PRED))

## 6  Figures 4 and 5: exact Shapley decomposition of the gated ensemble

Because both meta-learners are linear with non-negative coefficients and the gate selects one stream per city-day, the Shapley values of the composed forecast are the coefficient-weighted sums of the learners' exact tree Shapley values. Additivity is verified numerically.

In [ ]:
"""Stage D - exact Shapley values for the complete gated ensemble.

Because the meta-learner is linear with non-negative coefficients and the gate
selects one stream per city-day, the Shapley value of the whole forecast is the
coefficient-weighted sum of the learners' exact TreeSHAP values for the stream
the gate selected.  No sampling is involved, so the decomposition is exact and
additivity can be verified to machine precision.
"""
import sys, os, json, time
import numpy as np, pandas as pd
import shap

OUT = 'results'
RES = OUT
TAB = OUT
os.makedirs(TAB, exist_ok=True)
LAM, TAU_ = 4, 150
N_SEV, N_REST = 160, 160
NAMES = ("rf", "et", "gbr", "hgb")

PRETTY = {
 "lag0": "AQI today (t)", "lag1": "AQI at t-1", "lag2": "AQI at t-2", "lag3": "AQI at t-3",
 "lag4": "AQI at t-4", "lag5": "AQI at t-5", "lag6": "AQI at t-6", "lag13": "AQI at t-13",
 "lag_1y": "AQI one year ago", "roll_mean3": "3-day mean", "roll_mean7": "7-day mean",
 "roll_mean14": "14-day mean", "roll_mean30": "30-day mean", "roll_std7": "7-day s.d.",
 "roll_max7": "7-day maximum", "diff1": "Change since t-1", "diff7": "Change since t-7",
 "nat_mean": "National mean AQI", "igp_mean": "IGP mean AQI", "igp_mean_diff1": "IGP mean change",
 "city_minus_igp": "City minus IGP mean", "is_igp": "City in IGP", "city_mean_aqi": "City climatology",
 "pol_pm25": "PM2.5 lead pollutant", "pol_pm10": "PM10 lead pollutant", "pol_gas": "Gas lead pollutant",
 "obs_days7": "Observed days in last 7", "doy_sin": "Season (sin)", "doy_cos": "Season (cos)",
 "month": "Month", "weekday": "Weekday", "stubble_season": "Oct-Nov stubble season"}


def fitted_streams(D1):
    X = D1[FEATURES].astype(float)
    y = D1.target.values.astype(float)
    l0 = D1.lag0.values.astype(float)
    d = y - l0
    tr, va, te = [(D1.split == s).values for s in ("train", "val", "test")]
    trva = tr | va
    models = {"std": {}, "epi": {}}
    w = 1.0 + LAM * (y > 300)
    for stream, sw in (("std", None), ("epi", w)):
        for n in NAMES:
            t = time.time()
            models[stream][n] = make(n).fit(X[trva], d[trva],
                                            sample_weight=None if sw is None else sw[trva])
            print(f"   refit {stream} {n} {time.time()-t:.0f}s", flush=True)
    return models, X, y, l0, te


def main():
    D1 = pd.read_pickle(f'{OUT}/D1.pkl')
    meta = json.load(open(f"{RES}/run_h1.json"))
    import pickle
    cache = f"{OUT}/models_h1.pkl"
    if os.path.exists(cache):
        models, X, y, l0, te = pickle.load(open(cache, "rb"))
        print("loaded cached models", flush=True)
    else:
        models, X, y, l0, te = fitted_streams(D1)
        pickle.dump((models, X, y, l0, te), open(cache, "wb"))

    # verify the refit reproduces the saved predictions
    P = pd.read_csv(f"{RES}/predictions_h1.csv.gz")
    Xte = X[te]
    chk = {}
    for n in NAMES:
        chk[n] = float(np.abs(l0[te] + models["std"][n].predict(Xte) - P[LABEL[n]].values).max())
    print("refit vs saved, max abs deviation:", chk, flush=True)

    wS = np.array([meta["standard_weights"][n] for n in NAMES])
    wE = np.array([meta["episode_weights"][n] for n in NAMES])
    iS, iE = meta["standard_intercept"], meta["episode_intercept"]

    # explain a stratified sample: every Very Poor / Severe day plus a random slice
    sub = P.reset_index(drop=True)
    rng = np.random.default_rng(SEED)
    sev_all = np.where(sub.y.values > 300)[0]
    sev = rng.choice(sev_all, size=min(N_SEV, len(sev_all)), replace=False)
    rest = rng.choice(np.setdiff1d(np.arange(len(sub)), sev_all), size=N_REST, replace=False)
    idx = np.sort(np.concatenate([sev, rest]))
    Xs = Xte.iloc[idx]
    a_s = l0[te][idx]
    gate = a_s >= TAU_

    SV = {}
    for stream in ("std", "epi"):
        acc = np.zeros((len(idx), len(FEATURES)))
        base = 0.0
        ws = wS if stream == "std" else wE
        for k, n in enumerate(NAMES):
            m = models[stream][n]
            inner = m.est_ if hasattr(m, "est_") else m
            Xin = Xs.fillna(m.med_).fillna(0) if hasattr(m, "est_") else Xs
            ex = shap.TreeExplainer(inner)
            sv = ex.shap_values(Xin, check_additivity=False)
            acc += ws[k] * sv
            base += ws[k] * float(np.atleast_1d(ex.expected_value)[0])
            print(f"   shap {stream} {n} done", flush=True)
        SV[stream] = (acc, base + (iS if stream == "std" else iE))

    phi = np.where(gate[:, None], SV["epi"][0], SV["std"][0])
    base = np.where(gate, SV["epi"][1], SV["std"][1])
    delta_hat = np.where(gate,
                         sub["Episode stack"].values[idx] - a_s,
                         sub["Standard stack"].values[idx] - a_s)
    add_err = float(np.abs(phi.sum(1) + base - delta_hat).max())
    print("exact-Shapley additivity, max abs error:", add_err, flush=True)

    np.save(f"{TAB}/d_shap_values.npy", phi)
    np.save(f"{TAB}/d_shap_index.npy", idx)
    np.save(f"{TAB}/d_shap_base.npy", base)
    Xs.to_pickle(f"{TAB}/d_shap_X.pkl")

    imp = pd.Series(np.abs(phi).mean(0), index=[PRETTY.get(f, f) for f in FEATURES])
    imp.sort_values(ascending=False).round(4).to_csv(f"{TAB}/d_global_importance.csv")
    sev_mask = sub.y.values[idx] > 300
    comp = pd.DataFrame({"All explained days": np.abs(phi).mean(0),
                         "Very Poor / Severe days": np.abs(phi[sev_mask]).mean(0)},
                        index=[PRETTY.get(f, f) for f in FEATURES])
    comp.sort_values("All explained days", ascending=False).round(4).to_csv(f"{TAB}/d_importance_split.csv")
    json.dump({"additivity_max_abs_error": add_err,
               "n_explained": int(len(idx)), "n_severe_explained": int(sev_mask.sum()),
               "refit_max_abs_deviation": chk},
              open(f"{TAB}/d_shap_meta.json", "w"), indent=1)
    print("Stage D done", flush=True)


In [ ]:
if RUN_SHAP:
    main()

## 7  Figures

In [ ]:
"""All figures, rendered at 600 dpi with large, readable axis labels."""
import sys, os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

OUT = 'results'
RES = OUT
TAB = OUT
FIG = OUT
os.makedirs(FIG, exist_ok=True)
DPI = 600

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 13, "axes.titlesize": 14,
    "axes.labelsize": 13, "xtick.labelsize": 12, "ytick.labelsize": 12,
    "legend.fontsize": 12, "axes.linewidth": 0.9, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.6, "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03, "figure.facecolor": "white"})

RED, BLUE, GREY, GREEN, ORANGE = "#B22222", "#2A5CA5", "#8A8A8A", "#1F6354", "#C77B19"


def fig1_workflow(audit, meta):
    fig, ax = plt.subplots(figsize=(15.5, 3.5))
    ax.set_xlim(0, 100); ax.set_ylim(0, 30); ax.axis("off"); ax.grid(False)
    boxes = [
        ("CPCB daily AQI\nbulletins",
         f"Jun 2015 - 14 Oct 2026\n{audit['raw']:,} raw rows\n{audit['valid_city_days']:,} valid city-days",
         "#DCE8F7"),
        ("Cleaning and\nselection",
         f"Category-range check,\nname merge, de-duplication\n-> 63 cities, 2018-2026", "#DFEFDC"),
        ("Feature\nengineering",
         "32 features: lags, rolling\nstatistics, regional, city\ncontext, calendar;\ntarget = change in AQI", "#FBECD4"),
        ("Regime-aware\nstacking",
         "Standard stream +\nepisode stream (weight\n1+4 on AQI > 300);\ngate: AQI(t) >= 150", "#F3E0F2"),
        ("Output and\nevaluation",
         f"AQI at t+1, t+2, t+3\nTest {meta['n_test']:,} city-days\nExact Shapley values,\n80% conformal intervals", "#E6E6E6"),
    ]
    w, gap = 17.4, 2.6
    for i, (title, body, col) in enumerate(boxes):
        x = 1 + i * (w + gap)
        ax.add_patch(FancyBboxPatch((x, 2), w, 26, boxstyle="round,pad=0.3,rounding_size=1.2",
                                    linewidth=1.4, edgecolor="#333333", facecolor=col))
        ax.text(x + w / 2, 24.0, title, ha="center", va="center", fontsize=14.5, fontweight="bold")
        ax.text(x + w / 2, 12.5, body, ha="center", va="center", fontsize=11.0, linespacing=1.5)
        if i < len(boxes) - 1:
            ax.add_patch(FancyArrowPatch((x + w + 0.3, 15), (x + w + gap - 0.3, 15),
                                         arrowstyle="-|>", mutation_scale=18, linewidth=1.8,
                                         color="#333333"))
    fig.savefig(f"{FIG}/fig1_workflow.png", dpi=DPI)
    plt.close(fig)


def fig2_models(T1, Ps):
    order_all = T1.sort_values("RMSE")
    order_sev = T1.sort_values("RMSE AQI>300")
    fig, ax = plt.subplots(1, 3, figsize=(19, 6.4))

    def colr(idx):
        return [RED if m.startswith("RASE") else GREY if m == "Persistence" else BLUE for m in idx]

    b0 = ax[0].barh(order_all.index, order_all.RMSE, color=colr(order_all.index), height=0.72)
    ax[0].set_xlim(0, order_all.RMSE.max() * 1.20)
    ax[0].bar_label(b0, fmt="%.2f", padding=3, fontsize=10.5)
    ax[0].set_xlabel("RMSE on all test days (AQI points)")
    ax[0].set_title("(a) Overall accuracy")
    b1 = ax[1].barh(order_sev.index, order_sev["RMSE AQI>300"], color=colr(order_sev.index), height=0.72)
    ax[1].set_xlim(0, order_sev["RMSE AQI>300"].max() * 1.20)
    ax[1].bar_label(b1, fmt="%.1f", padding=3, fontsize=10.5)
    ax[1].axvline(T1.loc["Persistence", "RMSE AQI>300"], color=GREY, ls="--", lw=1.6)
    ax[1].set_xlabel("RMSE on Very Poor and Severe days, AQI > 300 (AQI points)")
    ax[1].set_title("(b) Accuracy on the most polluted days")
    hs = sorted(Ps)
    for m, mk, c in (("Persistence", "o", GREY), ("CatBoost", "s", BLUE),
                     ("Standard stack", "D", GREEN), ("RASE (proposed)", "^", RED)):
        v = [float(np.sqrt(np.mean((Ps[h].y - np.clip(Ps[h][m], 0, 500)) ** 2))) for h in hs]
        ax[2].plot(hs, v, marker=mk, color=c, lw=2.0, ms=9, label=m)
    ax[2].set_xticks(hs); ax[2].set_xlabel("Forecast horizon (days ahead)")
    ax[2].set_ylabel("RMSE (AQI points)"); ax[2].legend(frameon=False)
    ax[2].set_title("(c) Accuracy against horizon")
    for a in ax[:2]:
        a.grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(f"{FIG}/fig2_models.png", dpi=DPI)
    plt.close(fig)


def fig3_city_and_coverage(P, conf):
    fig, ax = plt.subplots(1, 2, figsize=(19, 5.6), gridspec_kw={"width_ratios": [1.65, 1]})
    k = (P[P.City == "Kanpur"].sort_values("TargetDate")
         .set_index("TargetDate").asfreq("D").reset_index())
    ax[0].plot(k.TargetDate, k.y, lw=1.4, color="#333333", label="Observed AQI")
    ax[0].plot(k.TargetDate, np.clip(k["RASE (proposed)"], 0, 500), lw=1.5, color=RED,
               label="RASE forecast, one day ahead")
    ax[0].axhspan(300, 500, color=ORANGE, alpha=0.12)
    ax[0].text(k.TargetDate.iloc[12], 420, "Very Poor and Severe", fontsize=11.5, color="#8a5a10")
    ax[0].set_ylabel("AQI"); ax[0].set_xlabel("Target date")
    ax[0].set_title("(a) Kanpur over the test period")
    ax[0].legend(frameon=False, loc="upper right")

    c = conf[conf.Subset.isin(CAT_NAMES)]
    cats = [x for x in CAT_NAMES if x in set(c.Subset)]
    pred_key = "Mondrian (by predicted category)"
    if pred_key not in set(c.Method):
        pred_key = "Mondrian (by issue-day category)"
    marg = [float(c[(c.Method == "Marginal") & (c.Subset == s)]["Coverage (%)"].iloc[0]) for s in cats]
    mond = [float(c[(c.Method == pred_key) & (c.Subset == s)]["Coverage (%)"].iloc[0]) for s in cats]
    x = np.arange(len(cats)); w = 0.38
    ax[1].bar(x - w / 2, marg, w, label="Marginal conformal", color=BLUE)
    ax[1].bar(x + w / 2, mond, w, label="Conformal, conditioned on predicted category", color=RED)
    ax[1].axhline(80, color="#333333", ls="--", lw=1.6)
    ax[1].text(len(cats) - 0.45, 81.5, "nominal 80%", fontsize=11.5, ha="right")
    ax[1].set_xticks(x); ax[1].set_xticklabels(cats, rotation=20, ha="right")
    ax[1].set_ylabel("Empirical coverage (%)"); ax[1].set_xlabel("Observed CPCB category")
    ax[1].set_ylim(0, 105); ax[1].legend(frameon=False, loc="lower left")
    ax[1].set_title("(b) Interval coverage by category")
    ax[1].grid(axis="x", visible=False)
    fig.tight_layout()
    fig.savefig(f"{FIG}/fig3_city_coverage.png", dpi=DPI)
    plt.close(fig)


def fig4_shap():
    phi = np.load(f"{TAB}/d_shap_values.npy")
    idx = np.load(f"{TAB}/d_shap_index.npy")
    Xs = pd.read_pickle(f"{TAB}/d_shap_X.pkl")
    comp = pd.read_csv(f"{TAB}/d_importance_split.csv", index_col=0)
    P = pd.read_csv(f"{RES}/predictions_h1.csv.gz")
    sev = P.y.values[idx] > 300
    top = comp.head(12).index[::-1]
    pretty = [PRETTY.get(f, f) for f in Xs.columns]
    pos = {p: j for j, p in enumerate(pretty)}

    fig, ax = plt.subplots(1, 2, figsize=(19, 7.2))
    yv = np.arange(len(top)); w = 0.4
    ax[0].barh(yv - w / 2, [comp.loc[t, "All explained days"] for t in top], w,
               color=BLUE, label="All explained days")
    ax[0].barh(yv + w / 2, [comp.loc[t, "Very Poor / Severe days"] for t in top], w,
               color=RED, label="Very Poor and Severe days")
    ax[0].set_yticks(yv); ax[0].set_yticklabels(top)
    ax[0].set_xlabel("Mean |Shapley value| (AQI points)")
    ax[0].set_title("(a) Contribution of each feature to the forecast change")
    ax[0].legend(frameon=False); ax[0].grid(axis="y", visible=False)

    show = list(top[::-1])[:10][::-1]
    rng = np.random.default_rng(0)
    for j, t in enumerate(show):
        col = pos[t]
        v = phi[:, col]; fv = Xs.iloc[:, col].values.astype(float)
        lo, hi = np.nanpercentile(fv, [5, 95])
        cvals = np.clip((fv - lo) / max(hi - lo, 1e-9), 0, 1)
        jit = rng.normal(0, 0.11, len(v))
        s = ax[1].scatter(v, j + jit, c=cvals, cmap="coolwarm", s=7, alpha=0.65, linewidths=0)
    ax[1].set_yticks(range(len(show))); ax[1].set_yticklabels(show)
    ax[1].axvline(0, color="#555555", lw=1.0)
    ax[1].set_xlabel("Shapley value: effect on the predicted change in AQI")
    ax[1].set_title("(b) How feature values move the forecast")
    cb = fig.colorbar(s, ax=ax[1], pad=0.01)
    cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"]); cb.set_label("Feature value")
    ax[1].grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(f"{FIG}/fig4_shap.png", dpi=DPI)
    plt.close(fig)


def fig5_waterfall():
    phi = np.load(f"{TAB}/d_shap_values.npy")
    idx = np.load(f"{TAB}/d_shap_index.npy")
    base = np.load(f"{TAB}/d_shap_base.npy")
    Xs = pd.read_pickle(f"{TAB}/d_shap_X.pkl")
    P = pd.read_csv(f"{RES}/predictions_h1.csv.gz", parse_dates=["TargetDate"])
    sub = P.iloc[idx].reset_index(drop=True)
    cand = sub[(sub.City == "Kanpur")]
    if len(cand) == 0:
        cand = sub
    j = int(cand.y.idxmax())
    pretty = np.array([PRETTY.get(f, f) for f in Xs.columns])
    v = phi[j]
    o = np.argsort(-np.abs(v))[:10]
    o = o[np.argsort(v[o])]
    fig, ax = plt.subplots(figsize=(12.5, 6.4))
    cols = [RED if x > 0 else BLUE for x in v[o]]
    ax.barh(np.arange(len(o)), v[o], color=cols, height=0.7)
    ax.set_yticks(np.arange(len(o)))
    ax.set_yticklabels([f"{pretty[k]}  =  {Xs.iloc[j, k]:.0f}" for k in o])
    ax.axvline(0, color="#555555", lw=1.0)
    ax.set_xlabel("Shapley value: effect on the predicted change in AQI")
    row = sub.iloc[j]
    ax.set_title(f"{row.City}, target {row.TargetDate.date()}:  observed AQI {row.y:.0f}, "
                 f"forecast {np.clip(row['RASE (proposed)'],0,500):.0f}\n"
                 f"baseline change {base[j]:+.1f} + contributions = predicted change")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    fig.savefig(f"{FIG}/fig5_waterfall.png", dpi=DPI)
    plt.close(fig)
    return dict(city=str(row.City), date=str(row.TargetDate.date()), y=float(row.y),
                pred=float(np.clip(row["RASE (proposed)"], 0, 500)), base=float(base[j]))


In [ ]:
if RUN_FIGURES:
    audit = json.load(open('results/audit.json'))
    meta = json.load(open('results/run_h1.json'))
    T = pd.read_csv('results/b1_main_multihorizon.csv', index_col=[0, 1])
    fig1_workflow(audit, meta)
    fig2_models(T.loc[1], PRED)
    fig3_city_and_coverage(PRED[1], pd.read_csv('results/c6_conformal.csv'))
    if RUN_SHAP:
        fig4_shap()
        json.dump(fig5_waterfall(), open('results/d_waterfall.json', 'w'), indent=1)
    print('figures written to results/')

## 8  Download every result

In [ ]:
import shutil
shutil.make_archive('RASE_v2_revised_results', 'zip', 'results')
try:
    from google.colab import files
    files.download('RASE_v2_revised_results.zip')
except ImportError:
    print('saved RASE_v2_revised_results.zip')